# Trading Agent — Fine-tuning Notebook

Fine-tunes **Llama 3.1 8B** on your paper trading decision dataset using **QLoRA + Unsloth**.
Exports a quantized **GGUF** model ready to run in Ollama on your Proxmox server.

### Requirements
- Runtime: **T4 GPU** (free Colab) or better
- Upload your `dataset_*.jsonl` file exported from the dashboard
- ~2–4h training on T4 for a typical dataset (500–3000 samples)

### Steps
1. Upload dataset → set `DATASET_PATH`
2. Run all cells
3. Download `trading-llm.Q4_K_M.gguf` at the end
4. Drop it into Ollama and set `LLM_PROVIDER=ollama` in the agent `.env`

## 1 · Install dependencies

In [ ]:
%%capture
# Unsloth: faster QLoRA with 80% less VRAM
!pip install unsloth
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Core training stack
!pip install transformers datasets trl peft bitsandbytes accelerate

# GGUF export
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121
!pip install huggingface_hub

## 2 · Config — edit these before running

In [ ]:
# ── Edit these ──────────────────────────────────────────────
DATASET_PATH   = "/content/dataset.jsonl"   # path after uploading
MODEL_NAME     = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
OUTPUT_DIR     = "/content/trading-llm"
GGUF_NAME      = "trading-llm.Q4_K_M.gguf"
MAX_SEQ_LEN    = 2048

# LoRA hyperparams (safe defaults for T4)
LORA_R         = 16      # rank — increase to 32 for more capacity if you have A100
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.05

# Training
EPOCHS         = 3
BATCH_SIZE     = 2       # per device — keep at 2 for T4
GRAD_ACCUM     = 4       # effective batch = 8
LR             = 2e-4
WARMUP_RATIO   = 0.05
# ────────────────────────────────────────────────────────────

print(f"Dataset : {DATASET_PATH}")
print(f"Base model: {MODEL_NAME}")
print(f"Output  : {OUTPUT_DIR}")

## 3 · GPU check

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU found — change runtime to T4 in Colab"
gpu = torch.cuda.get_device_properties(0)
print(f"GPU  : {gpu.name}")
print(f"VRAM : {gpu.total_memory / 1024**3:.1f} GB")
print(f"CUDA : {torch.version.cuda}")

## 4 · Load dataset

In [ ]:
import json
from datasets import Dataset

records = []
with open(DATASET_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

print(f"Loaded {len(records)} training samples")
if len(records) < 100:
    print("⚠  Very small dataset — consider running more paper trading cycles before training")
elif len(records) < 500:
    print("ℹ  Small dataset — model will learn basic patterns")
else:
    print("✓  Good dataset size")

# Preview a sample
print("\nSample record:")
print(json.dumps(records[0], indent=2)[:600], "...")

## 5 · Format into chat template

In [ ]:
SYSTEM_PROMPT = """You are a crypto trading agent operating a paper trading account.
You receive a market snapshot and current portfolio, then decide whether to buy, sell, or hold.

Rules:
- Never risk more than the configured max position per trade
- Prefer hold when signals are ambiguous
- Always justify your decision with clear, concise reasoning

Respond ONLY with a valid JSON object:
{"action": "buy"|"sell"|"hold", "asset": "BTC/USD", "amount_usd": 0, "confidence": 0.0, "reasoning": "..."}

No extra text, no markdown. Pure JSON only."""

def format_sample(record):
    """Convert Alpaca-format record into Llama 3 chat template."""
    user_content = record["input"] if record.get("input") else record.get("instruction", "")
    assistant_content = record["output"]

    return {
        "conversations": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_content},
            {"role": "assistant", "content": assistant_content},
        ]
    }

formatted = [format_sample(r) for r in records]

# 90/10 train/eval split
split = int(len(formatted) * 0.9)
train_data = Dataset.from_list(formatted[:split])
eval_data  = Dataset.from_list(formatted[split:])

print(f"Train : {len(train_data)} samples")
print(f"Eval  : {len(eval_data)} samples")

## 6 · Load base model with Unsloth

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LEN,
    dtype           = None,   # auto-detect
    load_in_4bit    = True,   # QLoRA
)

# Apply Llama 3 chat template
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

print(f"Model loaded — params: {sum(p.numel() for p in model.parameters()) / 1e9:.1f}B")

## 7 · Apply LoRA adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r                   = LORA_R,
    lora_alpha          = LORA_ALPHA,
    lora_dropout        = LORA_DROPOUT,
    target_modules      = ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
    bias                = "none",
    use_gradient_checkpointing = "unsloth",  # saves VRAM
    random_state        = 42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params : {trainable / 1e6:.1f}M  ({100 * trainable / total:.2f}% of total)")

## 8 · Tokenize dataset

In [ ]:
from unsloth.chat_templates import standardize_sharegpt, train_on_responses_only

def apply_template(examples):
    texts = [
        tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        for conv in examples["conversations"]
    ]
    return {"text": texts}

train_dataset = train_data.map(apply_template, batched=True)
eval_dataset  = eval_data.map(apply_template, batched=True)

print("Sample tokenized text (truncated):")
print(train_dataset[0]["text"][:400], "...")

## 9 · Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
import math

steps_per_epoch = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))
total_steps     = steps_per_epoch * EPOCHS

print(f"Steps per epoch : {steps_per_epoch}")
print(f"Total steps     : {total_steps}")

trainer = SFTTrainer(
    model       = model,
    tokenizer   = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
    args = SFTConfig(
        output_dir              = OUTPUT_DIR,
        num_train_epochs        = EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        learning_rate           = LR,
        warmup_ratio            = WARMUP_RATIO,
        lr_scheduler_type       = "cosine",
        optim                   = "adamw_8bit",   # memory efficient
        fp16                    = not torch.cuda.is_bf16_supported(),
        bf16                    = torch.cuda.is_bf16_supported(),
        logging_steps           = 10,
        eval_strategy           = "epoch",
        save_strategy           = "epoch",
        load_best_model_at_end  = True,
        metric_for_best_model   = "eval_loss",
        greater_is_better       = False,
        max_seq_length          = MAX_SEQ_LEN,
        dataset_text_field      = "text",
        packing                 = True,   # pack short sequences → better GPU utilisation
        report_to               = "none",
        seed                    = 42,
    ),
)

# Train only on assistant responses (not on prompt tokens)
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_header_id|>user<|end_header_id|>\n\n",
    response_part    = "<|start_header_id|>assistant<|end_header_id|>\n\n",
)

print("Starting training...")
trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"Runtime : {trainer_stats.metrics['train_runtime']:.0f}s  ({trainer_stats.metrics['train_runtime']/60:.1f} min)")
print(f"Final loss : {trainer_stats.metrics['train_loss']:.4f}")

## 10 · Evaluate & sanity-check the model

In [ ]:
import json

FastLanguageModel.for_inference(model)  # switch to inference mode (2x faster)

TEST_MARKET = {
    "market": {
        "BTC/USD": {"price": 82400, "change_24h": -2.1, "rsi_14": 38,
                    "high_24h": 85000, "low_24h": 81200, "volume_24h": 1200000000}
    },
    "portfolio": {"cash_usd": 8200, "positions": {}}
}

messages = [
    {"role": "system",    "content": SYSTEM_PROMPT},
    {"role": "user",      "content": json.dumps(TEST_MARKET)},
]

inputs = tokenizer.apply_chat_template(
    messages, tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

from transformers import TextStreamer
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print("Model response:")
print("─" * 60)
_ = model.generate(
    input_ids  = inputs,
    streamer   = streamer,
    max_new_tokens = 256,
    temperature = 0.1,
    do_sample   = True,
)
print("─" * 60)

## 11 · Export LoRA adapter (for future reuse)

In [ ]:
import os

adapter_path = os.path.join(OUTPUT_DIR, "lora-adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"LoRA adapter saved to {adapter_path}")
print("Keep this — you can resume training or merge later without rerunning from scratch")

## 12 · Export merged model as GGUF (Q4_K_M)

In [ ]:
# Merge LoRA weights into base model, then export quantized GGUF
# Q4_K_M = 4-bit quantization, ~4.5 GB — runs well on CPU via Ollama

gguf_path = os.path.join(OUTPUT_DIR, GGUF_NAME)

print("Merging LoRA into base model and exporting GGUF...")
print("This takes ~5–10 minutes on T4")

model.save_pretrained_gguf(
    OUTPUT_DIR,
    tokenizer,
    quantization_method = "q4_k_m",  # best quality/size tradeoff for CPU inference
)

# Rename to our model name
import glob, shutil
gguf_files = glob.glob(os.path.join(OUTPUT_DIR, "*.gguf"))
if gguf_files:
    shutil.move(gguf_files[0], gguf_path)
    size_gb = os.path.getsize(gguf_path) / 1024**3
    print(f"\n✓ GGUF exported: {gguf_path}")
    print(f"  File size: {size_gb:.2f} GB")
else:
    print("ERROR: no .gguf file found — check Unsloth output above")

## 13 · Download the model

In [ ]:
from google.colab import files
import os

gguf_path = os.path.join(OUTPUT_DIR, GGUF_NAME)

if os.path.exists(gguf_path):
    print(f"Downloading {GGUF_NAME} ({os.path.getsize(gguf_path)/1024**3:.2f} GB)...")
    print("Note: large file — download may take a few minutes")
    files.download(gguf_path)
else:
    print("GGUF file not found — run cell 12 first")

## 14 · Deploy to Ollama on Proxmox

After downloading the GGUF file, run these commands on your Proxmox server:

```bash
# 1. Install Ollama (if not already installed)
curl -fsSL https://ollama.com/install.sh | sh

# 2. Copy GGUF to Ollama models directory
mkdir -p ~/.ollama/models/blobs
cp trading-llm.Q4_K_M.gguf ~/.ollama/models/blobs/

# 3. Create a Modelfile
cat > Modelfile << 'EOF'
FROM ./trading-llm.Q4_K_M.gguf

SYSTEM """You are a crypto trading agent operating a paper trading account.
You receive a market snapshot and current portfolio, then decide whether to buy, sell, or hold.
Respond ONLY with a valid JSON object. No extra text, no markdown. Pure JSON only."""

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER num_predict 256
EOF

# 4. Register the model
ollama create trading-llm -f Modelfile

# 5. Test it
ollama run trading-llm '{"market":{"BTC/USD":{"price":82400,"change_24h":-2.1,"rsi_14":38}},"portfolio":{"cash_usd":8200,"positions":{}}}'

# 6. Update agent .env
# LLM_PROVIDER=ollama
# OLLAMA_BASE_URL=http://localhost:11434
# OLLAMA_MODEL=trading-llm
```

Expected inference speed on your Xeon W3550: **~3–6 tokens/sec** with Q4_K_M.  
For a trading agent running every 15 minutes, this is more than fast enough.

## 15 · Iterative retraining strategy

As the agent accumulates more data, retrain periodically:

```
Week 1-2   → ~100-300 samples  → baseline model
Week 3-4   → ~300-600 samples  → first meaningful improvement
Month 2+   → 1000+ samples     → model starts showing real edge
```

When retraining:
1. Export fresh dataset from dashboard (`POST /api/dataset/export`)
2. **Resume from LoRA adapter** (cell 6 → load `lora-adapter/` instead of base model)
   ```python
   # Change MODEL_NAME to your saved adapter path on Drive
   MODEL_NAME = "/content/drive/MyDrive/trading-llm/lora-adapter"
   ```
3. Rerun cells 8–13

Save the LoRA adapter to Google Drive between sessions to avoid retraining from scratch.

```python
from google.colab import drive
drive.mount('/content/drive')
shutil.copytree(adapter_path, '/content/drive/MyDrive/trading-llm/lora-adapter')
```